# TransMIL Full Colab Pipeline

This notebook runs the end-to-end TransMIL workflow on Google Colab:

1. Environment setup  
2. Dataset download (CAMELYON16 MIL-ready features)  
3. Train + test  
4. Evaluation plots (ROC/PR/confusion/calibration)  
5. Interpretability plots (token heatmaps)  
6. Optional coordinate heatmaps  
7. Optional top-k patch galleries  
8. Ablation and convergence comparison plots

In [ ]:
# Clone your repository (edit URL if needed)
%cd /content
!git clone https://github.com/szc19990412/TransMIL.git transmil
%cd /content/transmil

In [ ]:
# Install dependencies and Colab setup
!bash scripts/colab_setup.sh
!python -m pip install -r requirements.txt

In [ ]:
# Download CAMELYON16 MIL-ready features + optional split reference
!python scripts/download_camelyon16_torchmil.py --feature-set resnet50 --download-splits

In [ ]:
# Train
!python train.py --stage train --config Camelyon/TransMIL.yaml --gpus 0 --fold 0

In [ ]:
# Test (writes result.csv under logs/.../fold0)
!python train.py --stage test --config Camelyon/TransMIL.yaml --gpus 0 --fold 0

In [ ]:
# Pick checkpoint path automatically (best/latest epoch ckpt)
from pathlib import Path

ckpts = sorted(Path('logs/Camelyon/TransMIL/fold0').glob('epoch=*.ckpt'))
if not ckpts:
    raise FileNotFoundError('No epoch checkpoint found under logs/Camelyon/TransMIL/fold0')
CKPT_PATH = str(ckpts[-1])
print('Using checkpoint:', CKPT_PATH)

In [ ]:
# Evaluation plots: ROC/PR/confusion + calibration + confidence histogram + val curves
!python scripts/eval_test_and_plot.py \
  --config Camelyon/TransMIL.yaml \
  --ckpt "$CKPT_PATH" \
  --fold 0 --gpus 0

In [ ]:
# Interpretability: token importance vectors + grid heatmaps
!python scripts/visualize_transmil_interpretability.py \
  --config Camelyon/TransMIL.yaml \
  --ckpt "$CKPT_PATH" \
  --fold 0 --gpus 0 \
  --max-slides 20 --target-class 1

## Optional: Coordinate Heatmaps

Use this only if you have per-slide coordinate files with shape `[n_tokens, 2]` in:

- `Camelyon16/coords/<slide_id>.npy`

In [ ]:
# Optional coordinate-based heatmaps
!python scripts/visualize_transmil_interpretability.py \
  --config Camelyon/TransMIL.yaml \
  --ckpt "$CKPT_PATH" \
  --fold 0 --gpus 0 \
  --coords-dir Camelyon16/coords

## Optional: Top-k Patch Galleries

Use this only if you have a manifest CSV with columns:

- `slide_id`
- `patch_index`
- `image_path`

In [ ]:
# Optional top-k patch gallery
!python scripts/visualize_transmil_interpretability.py \
  --config Camelyon/TransMIL.yaml \
  --ckpt "$CKPT_PATH" \
  --fold 0 --gpus 0 \
  --patch-manifest Camelyon16/patch_manifest.csv \
  --top-k 16

## Optional: Ablation + Convergence Comparison

Create:

- `your_ablation_results.csv` with at least `experiment,auc`
- one or more Lightning `metrics.csv` paths to compare

In [ ]:
# Optional ablation and convergence plots
!python scripts/plot_ablation_and_convergence.py \
  --ablation-csv your_ablation_results.csv \
  --metrics-csvs logs/Camelyon/TransMIL/fold0/metrics.csv \
  --metrics-labels TransMIL \
  --metric-name auc \
  --out-dir plots

In [ ]:
# Preview generated PNG files
from pathlib import Path
from IPython.display import display, Image

pngs = sorted(Path('logs').rglob('*.png')) + sorted(Path('plots').rglob('*.png'))
print(f'Found {len(pngs)} PNG files')
for p in pngs[:20]:
    print(p)
    display(Image(filename=str(p), width=700))